In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np

In [ ]:
proj_path = "/mnt/1e99f03a-239b-4885-af31-60eb1322a5e1/IEL"
master_file_path = f"{proj_path}/sklearn/behaviour"

for subj in range(1, 39):

    # Subject id
    if subj < 10 :
        part_id = f"IEL00{subj}"
    else:
        part_id = f"IEL0{subj}"

    # skip removed subject
    if part_id == "IEL006":
        continue

    print(f"\nProcessing {part_id}")

    # Master file
    master_file = Path(master_file_path, f"{part_id}_master_file.csv")
    df = pd.read_csv(master_file)

    # Ouput folder
    output_dir = os.path.join(proj_path,"sklearn","data","individual",
    part_id,"regressors","activation","main_runs")

    os.makedirs(output_dir, exist_ok=True)

    # Subject index for scan runs

    if part_id == "IEL010":
        runs = [1, 2, 4]

    elif part_id == "IEL033":
        runs = [1, 2, 3]

    else:
        runs = [1, 2, 3, 4]

    # Experimental conditions
    conditions = [("Loss", "Positive"),
        ("Loss", "Neutral"),
        ("NoLoss", "Positive"),
        ("NoLoss", "Neutral")]
    
    # Creating experiment regressors
    for loss, emotion in conditions:

        # Condition data
        cond_df = df[(df["Loss"] == loss) &
            (df["Emotion"] == emotion)]

        
        for onset_type in ["Cue", "Task"]:

            onset_column = f"{onset_type}Onset"

            # Example:
            # cueLossPosOnset_001.txt

            file_name = (f"IEL_"
                f"{onset_type}"
                f"{loss}"
                f"{emotion[:3]}"
                f"Onset_{subj:03d}.txt")

            out_file = os.path.join(output_dir, file_name)

            # Write file
            with open(out_file, "w") as f:

                for run in runs:

                    # get only this run
                    run_df = cond_df[cond_df["Run"] == run]

                    # get onset values
                    onsets = run_df[onset_column].dropna().tolist()

                    # sort onset times
                    onsets.sort()

                    # if no trials in this run
                    if len(onsets) == 0:

                        f.write("*\n")

                    else:

                        # convert numbers to text
                        onset_text = []

                        for x in onsets:
                            onset_text.append(f"{x:.3f}")

                        # join into one line
                        line = " ".join(onset_text)

                        # write line
                        f.write(line + "\n")


print("\nDONE")



In [ ]:
%%writefile run_decon.csh
#!/bin/csh

# Paths
set proj_path = "/mnt/1e99f03a-239b-4885-af31-60eb1322a5e1/IEL"
set temp_path = "/media/cogemolab/home2/templates"
set proj = "IEL"

# Subject loop

foreach subj (001 002 003 004 005 007 008 009 010 011 012 013 014 015 016 017 018 019 020 \
021 022 023 024 025 026 027 028 029 030 031 032 033 034 035 036 037 038)

    set reg_path = $proj_path/sklearn/data/individual/${proj}${subj}/regressors/activation/main_runs

    set out_dir = $proj_path/sklearn/data/individual/${proj}${subj}/MVPA/BetaSer

    mkdir -p $out_dir

    cd $out_dir

    echo "Running LSS pipeline for ${proj}${subj}"

    # Condition loop

    foreach cond_idx (1 2 3 4)

        # Concat values
        if ($subj == 010 || $subj == 033) then

            set concat_val = "1D: 0 129 258"

        else

            set concat_val = "1D: 0 129 258 387"

        endif

        
        set st_im_1 = "-stim_times"
        set st_im_2 = "-stim_times"
        set st_im_3 = "-stim_times"
        set st_im_4 = "-stim_times"

        # Turn ON one condition to IM
        if ($cond_idx == 1) set st_im_1 = "-stim_times_IM"
        if ($cond_idx == 2) set st_im_2 = "-stim_times_IM"
        if ($cond_idx == 3) set st_im_3 = "-stim_times_IM"
        if ($cond_idx == 4) set st_im_4 = "-stim_times_IM"

        # Condition name
        if ($cond_idx == 1) set cond_name = "Cue_NoLoss_Positive"
        if ($cond_idx == 2) set cond_name = "Cue_Loss_Positive"
        if ($cond_idx == 3) set cond_name = "Cue_NoLoss_Neutral"
        if ($cond_idx == 4) set cond_name = "Cue_Loss_Neutral"

        echo "Running condition: ${cond_name}"

        # 3dDeconvolve
          3dDeconvolve \
            -overwrite \
            -input $proj_path/data/individual/${proj}${subj}/func/main_runs/${proj}${subj}_EP_TR_MNI_2mm_SI.nii.gz \
            -mask $temp_path/MNI152_T1_2mm_brain_GM_02182017.nii.gz \
            -concat "$concat_val" \
            -polort A \
            -nobout \
            -jobs 8 \
            -noFDR \
            -num_stimts 8 \
            -local_times \
            $st_im_1 1 $reg_path/${proj}_CueNoLossPosOnset_${subj}.txt 'GAM(8.6,0.547,1)' -stim_label 1 Cue_NoLoss_Positive \
            $st_im_2 2 $reg_path/${proj}_CueLossPosOnset_${subj}.txt 'GAM(8.6,0.547,1)' -stim_label 2 Cue_Loss_Positive \
            $st_im_3 3 $reg_path/${proj}_CueNoLossNeuOnset_${subj}.txt 'GAM(8.6,0.547,1)' -stim_label 3 Cue_NoLoss_Neutral \
            $st_im_4 4 $reg_path/${proj}_CueLossNeuOnset_${subj}.txt 'GAM(8.6,0.547,1)' -stim_label 4 Cue_Loss_Neutral \
            -stim_times 5 $reg_path/${proj}_TaskNoLossPosOnset_${subj}.txt 'GAM(8.6,0.547,0.5)' -stim_label 5 Task_NoLoss_Positive \
            -stim_times 6 $reg_path/${proj}_TaskLossPosOnset_${subj}.txt 'GAM(8.6,0.547,0.5)' -stim_label 6 Task_Loss_Positive \
            -stim_times 7 $reg_path/${proj}_TaskNoLossNeuOnset_${subj}.txt 'GAM(8.6,0.547,0.5)' -stim_label 7 Task_NoLoss_Neutral \
            -stim_times 8 $reg_path/${proj}_TaskLossNeuOnset_${subj}.txt 'GAM(8.6,0.547,0.5)' -stim_label 8 Task_Loss_Neutral \
            -ortvec $proj_path/data/individual/${proj}${subj}/func/main_runs/${proj}${subj}_MotionPar.txt'[1..6]' MotionParam \
            -ortvec $proj_path/data/individual/${proj}${subj}/func/main_runs/${proj}${subj}_MotionPar_derv.1D'[1..6]' MotionParamDerv \
            -censor $proj_path/data/individual/${proj}${subj}/func/main_runs/${proj}${subj}_censor.1D \
            -cbucket ./${proj}${subj}_TR_MNI_2mm_SI_censor_MR_betas.nii.gz \
            -x1D ./${proj}${subj}_TR_MNI_2mm_SI_censor_MR_deconLSS${cond_idx}.x1D \
            -x1D_stop \
            -xsave \
            -bucket ./${proj}${subj}_TR_MNI_2mm_SI_censor_MR_deconLSS${cond_idx}.nii.gz

        # 3dLSS
          3dLSS \
            -matrix ${proj}${subj}_TR_MNI_2mm_SI_censor_MR_deconLSS${cond_idx}.x1D \
            -input $proj_path/data/individual/${proj}${subj}/func/main_runs/${proj}${subj}_EP_TR_MNI_2mm_SI.nii.gz \
            -mask $temp_path/MNI152_T1_2mm_brain_GM_02182017.nii.gz \
            -prefix ${proj}${subj}_TR_MNI_2mm_SI_MR_deconLSS_betas${cond_idx}.nii.gz

        # Create trial wise NIFTI files
        set lss_file = ${proj}${subj}_TR_MNI_2mm_SI_MR_deconLSS_betas${cond_idx}.nii.gz

        # Number of subbricks
        set n_bricks = `3dinfo -nv $lss_file`

        # Output directory
        set cond_outdir = \
        $proj_path/sklearn/data/individual/${proj}${subj}/BetaSeries/${cond_name}

        mkdir -p $cond_outdir

        echo "Extracting ${n_bricks} trial betas"

        # Extract each subbrick
        
        # Last subbrick index
        @ last_brick = $n_bricks - 1

        # Loop through all subbricks
        foreach brick (`seq 0 $last_brick`)

            echo "Extracting ${cond_name}_${brick}"

            3dcalc \
                -a "${lss_file}[${brick}]" \
                -expr 'a' \
                -prefix ${cond_outdir}/${cond_name}_${brick}.nii.gz

        end

    end

    echo "Finished subject ${proj}${subj}"

end

echo "All subjects complete"


Overwriting run_decon.csh


In [ ]:
!chmod +x run_decon.csh 
!tcsh run_decon.csh